In [6]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs 
import threading
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service


options = Options()
service = Service(ChromeDriverManager().install())

lock = threading.Lock()

def scrape_pages(urls):
	driver = webdriver.Chrome(service=service, options=options)
	for url in urls:
		driver.get(url)
		soup = bs(driver.page_source, 'html.parser')
		products = []
		texts = []
		products = soup.find_all(class_="full-unstyled-link")
		text = soup.get_text(separator="\n", strip=True).split(sep="\n")
		products = list(set([product.get_text(strip=True) for product in products]))
		texts = list(set([t for t in text if t not in products and t != '']))
		with lock:
			with open('products_factorybuy.txt', 'a') as f:
				for product in products:
					f.write(f"{product}\ttrue\n")
			with open('negatifs_factorybuy.txt.txt', 'a') as f:
				for t in texts:
					f.write(f"{t}\tfalse\n")
		
	driver.quit()

In [7]:
threads = []

for i in range(0, 5):
	base_url = "https://www.factorybuy.com.au/collections/all?page="
	urls = [f"{base_url}{i}" for i in range(i*20+1,(i+1)*20)]
	t = threading.Thread(target=scrape_pages, args=(urls,))
	threads.append(t)
	t.start()
